In [3]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_community.document_loaders import PyPDFLoader
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import openai
import chromadb
from chromadb.config import Settings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from openai import OpenAI
import os

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))

model = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)
EMBEDDING_API_KEY = os.getenv('EMBEDDING_API_KEY', '')
EMBEDDING_DEPLOYMENT_NAME = os.getenv('EMBEDDING_DEPLOYMENT_NAME', 'BT-Embedding')
EMBEDDING_ENDPOINT = os.getenv('EMBEDDING_ENDPOINT', '')
DB_PATH = "./chroma_db"

# Descriptive collection names — helps the agent decide which to query
COLLECTION_NAME_ATTENTION     = "attention_is_all_you_need"   # paper-1.pdf
COLLECTION_NAME_OPENAI_REPORT = "openai_technical_report"  
# -------------------
# 2. Clients
# -------------------
azure_client = OpenAI(
    base_url=EMBEDDING_ENDPOINT,
    api_key=EMBEDDING_API_KEY
)

chroma_client = chromadb.PersistentClient(path=DB_PATH)
# collection_1 = chroma_client.get_or_create_collection(name=COLLECTIONA_NAME_1)

# -------------------
# 3. Embedding Function
# -------------------
# -------------------
# 3. Embedding Function
# -------------------
def get_embedding(text: str) -> list:
    response = azure_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOYMENT_NAME,
    )
    return response.data[0].embedding

# -------------------
# 4. Generic Ingest Helper
# -------------------
def ingest_pdfs(
    pdf_files: list,
    collection_name: str,
    id_prefix: str,
    chunk_size: int = 800,
    chunk_overlap: int = 200,
    batch_size: int = 50,
):
    """
    Load PDFs -> split -> embed -> store in a named Chroma collection.

    Args:
        pdf_files:       List of PDF file paths to ingest.
        collection_name: Chroma collection to store chunks in.
        id_prefix:       Unique prefix for chunk IDs (avoids collisions across collections).
        chunk_size:      Characters per chunk.
        chunk_overlap:   Overlap between consecutive chunks.
        batch_size:      Number of chunks per embedding batch (avoids rate limits).
    """
    # --- Load ---
    docs_raw = []
    for pdf in pdf_files:
        try:
            loader = PyPDFLoader(pdf)
            docs_raw.extend(loader.load())
            print(f"  [load]  Loaded: {pdf}")
        except Exception as e:
            print(f"  [load]  Failed: {pdf} -> {e}")

    if not docs_raw:
        print(f"  [warn]  No documents loaded for collection '{collection_name}'. Skipping.")
        return

    print(f"  [load]  Total pages: {len(docs_raw)}")

    # --- Split ---
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    splits = splitter.split_documents(docs_raw)
    print(f"  [split] Total chunks: {len(splits)}")

    # --- Embed + Store ---
    collection = chroma_client.get_or_create_collection(name=collection_name)

    for i in range(0, len(splits), batch_size):
        batch      = splits[i : i + batch_size]
        # Prefixed IDs -> unique across collections, no collisions
        ids        = [f"{id_prefix}_chunk_{i + j}" for j in range(len(batch))]
        texts      = [doc.page_content for doc in batch]
        metadatas  = [doc.metadata for doc in batch]
        embeddings = [get_embedding(text) for text in texts]

        collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas,
        )
        print(
            f"  [store] Batch {i // batch_size + 1} — "
            f"chunks {i} to {min(i + batch_size, len(splits))} stored."
        )

    print(f"  [done]  {len(splits)} chunks in collection '{collection_name}' at '{DB_PATH}'\n")

# -------------------
# 5. Ingest Both Papers
# -------------------
print("=== Ingesting: Attention Is All You Need (paper-1.pdf) ===")
ingest_pdfs(
    pdf_files=["paper-1.pdf"],
    collection_name=COLLECTION_NAME_ATTENTION,
    id_prefix="attention",
)

print("=== Ingesting: OpenAI Technical Report (paper-3.pdf) ===")
ingest_pdfs(
    pdf_files=["paper-3.pdf"],
    collection_name=COLLECTION_NAME_OPENAI_REPORT,
    id_prefix="openai_report",
)

print("All collections ready. Next step: build the agent with tool routing.")


=== Ingesting: Attention Is All You Need (paper-1.pdf) ===
  [load]  Loaded: paper-1.pdf
  [load]  Total pages: 15
  [split] Total chunks: 71
  [store] Batch 1 — chunks 0 to 50 stored.
  [store] Batch 2 — chunks 50 to 71 stored.
  [done]  71 chunks in collection 'attention_is_all_you_need' at './chroma_db'

=== Ingesting: OpenAI Technical Report (paper-3.pdf) ===
  [load]  Loaded: paper-3.pdf
  [load]  Total pages: 100
  [split] Total chunks: 494
  [store] Batch 1 — chunks 0 to 50 stored.
  [store] Batch 2 — chunks 50 to 100 stored.
  [store] Batch 3 — chunks 100 to 150 stored.


KeyboardInterrupt: 

In [25]:
query = "What is Reinforcement Learning & Alignment?"
query_embedding = get_embedding(query)


collection = chroma_client.get_collection(name=COLLECTION_NAME_ATTENTION)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

for i, doc in enumerate(results['documents'][0]):
    source = results['metadatas'][0][i].get('source', 'unknown')
    print(f"\nChunk {i+1} (source: {source}):")
    print(doc)


Chunk 1 (source: paper-1.pdf):
recurrent nets: the difficulty of learning long-term dependencies, 2001.
[13] Sepp Hochreiter and Jürgen Schmidhuber. Long short-term memory. Neural computation,
9(8):1735–1780, 1997.
[14] Zhongqiang Huang and Mary Harper. Self-training PCFG grammars with latent annotations
across languages. In Proceedings of the 2009 Conference on Empirical Methods in Natural
Language Processing, pages 832–841. ACL, August 2009.
[15] Rafal Jozefowicz, Oriol Vinyals, Mike Schuster, Noam Shazeer, and Yonghui Wu. Exploring
the limits of language modeling. arXiv preprint arXiv:1602.02410, 2016.
[16] Łukasz Kaiser and Samy Bengio. Can active memory replace attention? In Advances in Neural
Information Processing Systems, (NIPS), 2016.

Chunk 2 (source: paper-1.pdf):
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
tra

In [6]:
from langchain.tools.retriever import create_retriever_tool
from langchain_community.vectorstores import Chroma
from langchain_core.embeddings import Embeddings
class AzureCustomEmbeddings(Embeddings):
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [get_embedding(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return get_embedding(text)

embedding_function = AzureCustomEmbeddings()

# -------------------
# 7. Wrap Collections as LangChain Vectorstores
# -------------------
vectorstore_attention = Chroma(
    client=chroma_client,
    collection_name=COLLECTION_NAME_ATTENTION,
    embedding_function=embedding_function,
)

vectorstore_openai = Chroma(
    client=chroma_client,
    collection_name=COLLECTION_NAME_OPENAI_REPORT,
    embedding_function=embedding_function,
)

# -------------------
# 8. Retrievers
# -------------------
retriever_attention = vectorstore_attention.as_retriever(search_kwargs={"k": 3})
retriever_openai    = vectorstore_openai.as_retriever(search_kwargs={"k": 3})

# -------------------
# 9. Retriever Tools
# -------------------
tool_attention = create_retriever_tool(
    retriever_attention,
    "retriever_attention_is_all_you_need",
    "Search and retrieve information about the Transformer architecture, "
    "attention mechanism, multi-head attention, positional encoding, "
    "encoder-decoder structure from the 'Attention Is All You Need' paper.",
)

tool_openai = create_retriever_tool(
    retriever_openai,
    "retriever_openai_technical_report",
    "Search and retrieve information about GPT-4, RLHF, reinforcement learning "
    "from human feedback, alignment, safety, model capabilities and evaluations "
    "from the OpenAI technical report.",
)

tools = [tool_attention, tool_openai]




C:\Users\303370\AppData\Local\Temp\ipykernel_41120\994433965.py:16: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore_attention = Chroma(


In [21]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict

from langchain_core.messages import BaseMessage

from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:

def agent(state):
     
     """
    Invokes the agent model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply end.

    Args:
        state (messages): The current state

    Returns:
        dict: The updated state with the agent response appended to messages
    """
     
     print("Call_agent")
     messages = state["messages"]
     model_with_tools = model.bind_tools(tools)  
     response = model_with_tools.invoke(messages)
     return {"messages": [response]}

        

In [23]:
from typing import Annotated, Literal, Sequence
from typing_extensions import TypedDict

from langchain import hub
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

from pydantic import BaseModel, Field

In [ ]:

### Edges
def grade_documents(state) -> Literal["generate", "rewrite"]:
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (messages): The current state

    Returns:
        str: A decision for whether the documents are relevant or not
    """

    print("---CHECK RELEVANCE---")

    class grade(BaseModel):
        """Binary score for relevance check."""

        binary_score: str = Field(description="Relevance score 'yes' or 'no'")

  
    llm_with_tool = model.with_structured_output(grade)

    # Prompt
    prompt = PromptTemplate(
        template="""You are a grader assessing relevance of a retrieved document to a user question. \n 
        Here is the retrieved document: \n\n {context} \n\n
        Here is the user question: {question} \n
        If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
        Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.""",
        input_variables=["context", "question"],
    )
    chain = prompt | llm_with_tool

    messages = state["messages"]
    last_message = messages[-1]

    question = messages[0].content
    docs = last_message.content

    scored_result = chain.invoke({"question": question, "context": docs})

    score = scored_result.binary_score

    if score == "yes":
        print("---DECISION: DOCS RELEVANT---")
        return "generate"

    else:
        print("---DECISION: DOCS NOT RELEVANT---")
        print(score)
        return "rewrite"

In [ ]:

def generate(state):
    """
    Generate answer

    Args:
        state (messages): The current state

    Returns:
         dict: The updated message
    """
    print("---GENERATE---")
    messages = state["messages"]
    question = messages[0].content
    last_message = messages[-1]

    docs = last_message.content

    prompt = hub.pull("rlm/rag-prompt")


    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = prompt | model | StrOutputParser()

  
    response = rag_chain.invoke({"context": docs, "question": question})
    return {"messages": [response]}

In [26]:

def rewrite(state):
    """
    Transform the query to produce a better question.

    Args:
        state (messages): The current state

    Returns:
        dict: The updated state with re-phrased question
    """

    print("---TRANSFORM QUERY---")
    messages = state["messages"]
    question = messages[0].content

    msg = [
        HumanMessage(
            content=f""" \n 
    Look at the input and try to reason about the underlying semantic intent / meaning. \n 
    Here is the initial question:
    \n ------- \n
    {question} 
    \n ------- \n
    Formulate an improved question: """,
        )
    ]

    # Gradera
    response = model.invoke(msg)
    return {"messages": [response]}

In [ ]:
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition


workflow = StateGraph(AgentState)

workflow.add_node("agent", agent)  
retrieve = ToolNode([tool_attention,tool_openai])
workflow.add_node("retrieve", retrieve)
workflow.add_node("rewrite", rewrite) 
workflow.add_node(
    "generate", generate
) 
workflow.add_edge(START, "agent")

workflow.add_conditional_edges(
    "agent",
    
    tools_condition,
    {
        
        "tools": "retrieve",
        END: END,
    },
)

workflow.add_conditional_edges(
    "retrieve",
   
    grade_documents,
)
workflow.add_edge("generate", END)
workflow.add_edge("rewrite", "agent")

# Compile
graph = workflow.compile()



In [32]:
graph.invoke({"messages": [HumanMessage(content="GPT-4 Technical Report")]})

Call_agent


{'messages': [HumanMessage(content='GPT-4 Technical Report', additional_kwargs={}, response_metadata={}, id='c9166e09-b725-4f95-a676-4a4813de1c28'),
  AIMessage(content='The **GPT-4 Technical Report** is OpenAI’s report describing GPT-4’s development, capabilities, limitations, and safety work.\n\nKey points:\n- **GPT-4** is a large multimodal model that can accept **image and text inputs** and produce **text outputs**.\n- It was evaluated on a wide range of **professional and academic benchmarks**, often performing at or above human-level on some standardized tests.\n- The report emphasizes **predictable scaling**: OpenAI studied how performance and loss scale with compute and model size.\n- It discusses **alignment and safety mitigations**, including adversarial testing and model behavior improvements.\n- The report is intentionally limited in some technical details, especially about:\n  - model size\n  - architecture\n  - hardware\n  - training compute\n  - dataset construction\n\nC